# `12 — Bellman–Ford algorithm (negative edges allowed)`

## **When to use**
- Weighted graph **may contain negative edges**.
- Shortest paths from source s to all vertices.
- Can **detect negative cycles** reachable from s.

## **Complexity**
| Result | Time | Memory |
|---|---:|---:|
| Bellman–Ford | **O(V·E)** | O(V) |

## **DP interpretation (why V-1 iterations)**
Let $d^{(l)}[v]$ = best distance from $s$ to $v$ using at most $l$ edges.
Then:
$$d^{(l)}[u] = \min_{\text{edges } (v \to u)} \left( d^{(l-1)}[v] + w(v,u) \right)$$

If there is no negative cycle, an optimal path never needs $\geq V$ edges
(because a repeated vertex creates a cycle that can be removed if its weight $\geq 0$).
So after **V-1** passes, distances stabilize at shortest paths.

## **Negative cycle detection**
After V-1 relax rounds, do one more round.
If any distance can still be improved, a negative cycle exists.


In [ ]:
from typing import List, Tuple, Optional


Edge = Tuple[int, int, float]


def bellman_ford_visual(n: int, edges: List[Edge], *, s: int) -> Tuple[Optional[List[float]], Optional[List[Optional[int]]]]:

    INF: float = float("inf")
    d: List[float] = [INF] * n
    parent: List[Optional[int]] = [None] * n
    d[s] = 0.0

    print("-" * 80)
    print(f"Bellman–Ford from s={s}")
    print("-" * 80)
    print("We perform V-1 full relax passes over all edges.")
    print("If on the V-th pass something improves => negative cycle exists.")
    print("-" * 80)

    # V-1 iterations:
    for it in range(n - 1):
        changed: List[Tuple[int, float, float, int]] = []  # (u, old, new, via v)
        print(f"\nPass {it+1}/{n-1}")
        for v, u, w in edges:
            if d[v] == INF:
                continue
            new_d = d[v] + w
            if new_d < d[u]:
                changed.append((u, d[u], new_d, v))
                d[u] = new_d
                parent[u] = v

        print("d:", d)
        if changed:
            print("Updates in this pass:")
            for u, old, new, v in changed:
                print(f"  d[{u}] : {old} -> {new} via {v}->{u}")
        else:
            print("No updates -> early stop (already stable).")
            break

    # Extra pass to detect negative cycle:
    print("\n" + "-" * 80)
    print("Negative cycle check: one more relax pass")
    print("-" * 80)

    for v, u, w in edges:
        if d[v] == INF:
            continue
        if d[v] + w < d[u]:
            print(f"✅ Improvement still possible on edge {v}->{u} (w={w})")
            print("=> Negative cycle exists (reachable from source).")
            return None, None

    print("No improvement on extra pass => no negative cycle.")
    return d, parent


In [ ]:
# Example 01: Graph with a negative edge but no negative cycle.

n = 5
edges_ok: List[Edge] = [
    (0, 1, 5),
    (0, 2, 2),
    (2, 1, -1),
    (1, 3, 3),
    (2, 3, 4),
    (3, 4, 1),
]

d, parent = bellman_ford_visual(n, edges_ok, s=0)
print("\nResult:", d)

--------------------------------------------------------------------------------
Bellman–Ford from s=0
--------------------------------------------------------------------------------
We perform V-1 full relax passes over all edges.
If on the V-th pass something improves => negative cycle exists.
--------------------------------------------------------------------------------

Pass 1/4
d: [0.0, 1.0, 2.0, 4.0, 5.0]
Updates in this pass:
  d[1] : inf -> 5.0 via 0->1
  d[2] : inf -> 2.0 via 0->2
  d[1] : 5.0 -> 1.0 via 2->1
  d[3] : inf -> 4.0 via 1->3
  d[4] : inf -> 5.0 via 3->4

Pass 2/4
d: [0.0, 1.0, 2.0, 4.0, 5.0]
No updates -> early stop (already stable).

--------------------------------------------------------------------------------
Negative cycle check: one more relax pass
--------------------------------------------------------------------------------
No improvement on extra pass => no negative cycle.

Result: [0.0, 1.0, 2.0, 4.0, 5.0]


In [3]:
# Example 02: Graph with a reachable negative cycle:
# 1 -> 2 -> 3 -> 1 total weight = -2

n = 4
edges_neg_cycle: List[Edge] = [
    (0, 1, 1),
    (1, 2, -1),
    (2, 3, -1),
    (3, 1, 0),
]

d, parent = bellman_ford_visual(n, edges_neg_cycle, s=0)
print("\nResult:", d)

--------------------------------------------------------------------------------
Bellman–Ford from s=0
--------------------------------------------------------------------------------
We perform V-1 full relax passes over all edges.
If on the V-th pass something improves => negative cycle exists.
--------------------------------------------------------------------------------

Pass 1/3
d: [0.0, -1.0, 0.0, -1.0]
Updates in this pass:
  d[1] : inf -> 1.0 via 0->1
  d[2] : inf -> 0.0 via 1->2
  d[3] : inf -> -1.0 via 2->3
  d[1] : 1.0 -> -1.0 via 3->1

Pass 2/3
d: [0.0, -3.0, -2.0, -3.0]
Updates in this pass:
  d[2] : 0.0 -> -2.0 via 1->2
  d[3] : -1.0 -> -3.0 via 2->3
  d[1] : -1.0 -> -3.0 via 3->1

Pass 3/3
d: [0.0, -5.0, -4.0, -5.0]
Updates in this pass:
  d[2] : -2.0 -> -4.0 via 1->2
  d[3] : -3.0 -> -5.0 via 2->3
  d[1] : -3.0 -> -5.0 via 3->1

--------------------------------------------------------------------------------
Negative cycle check: one more relax pass
------------------

**Real-world mapping**: currency arbitrage: negative cycle means you can keep improving profit indefinitely.
